In [1]:
# !pip -q install feedparser beautifulsoup4 lxml readability-lxml newspaper3k trafilatura tqdm

In [2]:
import time
import re
import hashlib
from datetime import datetime, timezone
from urllib.parse import urlparse

import requests
import feedparser
import pandas as pd
from tqdm.auto import tqdm

from bs4 import BeautifulSoup
from readability import Document
import trafilatura
from newspaper import Article
import os


/Users/zzz/Documents/MIE1624/Group_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}
TIMEOUT = 20
SLEEP_BETWEEN_REQUESTS = 1.0

RSS_FEEDS = {
    # "New York Times": "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
    # "Reuters": "https://feeds.reuters.com/reuters/topNews",
    "The Guardian": "https://www.theguardian.com/world/rss",
    "BBC News": "https://feeds.bbci.co.uk/news/rss.xml",
    "Global News": "https://globalnews.ca/feed/",
    "Sky News": "https://feeds.skynews.com/feeds/rss/home.xml",
    "Deutsche Welle World": "https://rss.dw.com/rdf/rss-en-world",
    "TechCrunch": "https://techcrunch.com/feed/",
    # "Yahoo Finance Top": "https://finance.yahoo.com/news/rss",
    # "CNN": "http://rss.cnn.com/rss/cnn_topstories.rss",
    # "Fox News": "http://feeds.foxnews.com/foxnews/latest",
}

ITEMS_PER_SOURCE = 100

In [4]:
def safe_get(url: str) -> requests.Response | None:
    """GET with headers, timeout, and basic error handling."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True)
        if resp.status_code == 200 and resp.content:
            return resp
        return None
    except requests.RequestException:
        return None

def clean_text(text: str) -> str:
    """Normalize whitespace; keep paragraphs readable."""
    if not text:
        return ""
    text = re.sub(r"\r|\t", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \xa0]{2,}", " ", text)
    return text.strip()


In [5]:
def extract_with_trafilatura(url: str) -> str | None:
    try:
        downloaded = trafilatura.fetch_url(url, no_ssl=True)
        if downloaded:
            extracted = trafilatura.extract(
                downloaded,
                include_comments=False,
                include_images=False,
                include_tables=False
            )
            if extracted and len(extracted.split()) > 50:
                return extracted
    except Exception:
        pass
    return None

def extract_with_readability(html: str) -> str | None:
    try:
        doc = Document(html)
        content_html = doc.summary(html_partial=True)
        if content_html:
            soup = BeautifulSoup(content_html, "lxml")
            # Remove scripts/styles/navs
            for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
                tag.decompose()
            text = "\n".join(p.get_text(" ", strip=True) for p in soup.find_all(["p", "li"]))
            if text and len(text.split()) > 50:
                return text
    except Exception:
        pass
    return None

def extract_with_newspaper(url: str) -> str | None:
    try:
        art = Article(url)
        art.download()
        art.parse()
        text = art.text
        if text and len(text.split()) > 50:
            return text
    except Exception:
        pass
    return None

def extract_with_bs4_basic(html: str) -> str | None:
    try:
        soup = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
        text = "\n".join(p for p in paras if p)
        if text and len(text.split()) > 50:
            return text
    except Exception:
        pass
    return None

def extract_article_text(url: str) -> str:
    # trafilatura (no HTML required)
    text = extract_with_trafilatura(url)
    if text:
        return clean_text(text)

    # Get HTML once for readability/BS4
    resp = safe_get(url)
    if resp is None:
        return ""

    html = resp.text or ""

    # readability (needs HTML)
    text = extract_with_readability(html)
    if text:
        return clean_text(text)

    # newspaper3k
    text = extract_with_newspaper(url)
    if text:
        return clean_text(text)

    # BS4 naive paragraph join
    text = extract_with_bs4_basic(html)
    if text:
        return clean_text(text)

    return ""  # give up


In [6]:
def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", "ignore")).hexdigest()

def parse_authors(entry) -> str:
    # Feedparser puts authors in various places
    if hasattr(entry, "author") and entry.author:
        return entry.author
    if hasattr(entry, "authors") and entry.authors:
        # authors is often list of dicts with 'name'
        names = []
        for a in entry.authors:
            if isinstance(a, dict) and a.get("name"):
                names.append(a["name"])
            elif hasattr(a, "name"):
                names.append(a.name)
        if names:
            return ", ".join(names)
    return ""

def parse_published(entry) -> str:
    # Return ISO-like string if possible
    if hasattr(entry, "published"):
        return entry.published
    if hasattr(entry, "updated"):
        return entry.updated
    return ""

In [8]:
records = []
seen_links = set()

for source_name, feed_url in RSS_FEEDS.items():
    print(f"\nFetching RSS: {source_name} — {feed_url}")
    feed = feedparser.parse(feed_url)

    if feed.bozo:
        print(f"Feed parsing had minor issues (bozo={feed.bozo}). Proceeding anyway.")

    entries = feed.entries[:ITEMS_PER_SOURCE] if hasattr(feed, "entries") else []
    print(f"  Found {len(entries)} entries (taking up to {ITEMS_PER_SOURCE}).")

    for entry in tqdm(entries, desc=f"Scraping {source_name}", leave=False):
        title = getattr(entry, "title", "").strip()
        link = getattr(entry, "link", "").strip()
        description = getattr(entry, "summary", getattr(entry, "description", "")).strip()
        published = parse_published(entry)
        entry_id = getattr(entry, "id", getattr(entry, "guid", link))
        authors = parse_authors(entry)

        if not link or link in seen_links:
            continue
        seen_links.add(link)

        # Fetch article text
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        text = extract_article_text(link)

        # Normalize common fields
        records.append({
            "source": source_name,
            "title": title,
            "link": link,
            "description": clean_text(BeautifulSoup(description, "lxml").get_text(" ", strip=True)),
            "published": published,
            "entry_id": entry_id or sha1(link),
            "authors": authors,
            "domain": urlparse(link).netloc,
            "fetched_at": datetime.now(timezone.utc).isoformat(),
            "text": text
        })


Fetching RSS: The Guardian — https://www.theguardian.com/world/rss
  Found 45 entries (taking up to 100).



Fetching RSS: BBC News — https://feeds.bbci.co.uk/news/rss.xml
  Found 39 entries (taking up to 100).



Fetching RSS: Global News — https://globalnews.ca/feed/
  Found 10 entries (taking up to 100).



Fetching RSS: Sky News — https://feeds.skynews.com/feeds/rss/home.xml
  Found 10 entries (taking up to 100).



Fetching RSS: Deutsche Welle World — https://rss.dw.com/rdf/rss-en-world
  Found 13 entries (taking up to 100).



Fetching RSS: TechCrunch — https://techcrunch.com/feed/
  Found 20 entries (taking up to 100).


In [13]:
DATA_DIR = "/Users/zzz/Documents/MIE1624/Group_Project/data/vectorstore_news_ai"

df = pd.DataFrame.from_records(records)

# Drop exact-duplicate links if any slipped through
if not df.empty:
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)

print(f"\nCollected {len(df)} articles total.")
display_cols = ["source", "title", "published", "authors", "domain", "link"]
display(df[display_cols].head(15))

# --- Save CSV into data folder ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"news_articles_combined_{timestamp}.csv"

# Join path safely
combined_path = os.path.join(DATA_DIR, filename)

# Save
df.to_csv(combined_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Saved combined CSV to: {combined_path}")


Collected 136 articles total.


,source,title,published,authors,domain,link
0,The Guardian,TikTok influencer killed in public ‘execution’...,"Tue, 11 Nov 2025 14:07:11 GMT",Eromo Egbejule West Africa correspondent,www.theguardian.com,https://www.theguardian.com/world/2025/nov/11/...
1,The Guardian,Violent reprisals after DRC whistleblowers dis...,"Tue, 11 Nov 2025 08:00:12 GMT",Josephine Moulds and Sonia Rolley,www.theguardian.com,https://www.theguardian.com/environment/2025/n...
2,The Guardian,US has sent $7.5m to Equatorial Guinea to acce...,"Mon, 10 Nov 2025 23:18:05 GMT",Andrew Roth and Joseph Gedeon in Washington,www.theguardian.com,https://www.theguardian.com/us-news/2025/nov/1...
3,The Guardian,Protesters target major new Nigerian museum em...,"Mon, 10 Nov 2025 15:19:20 GMT",Eromo Egbejule West Africa correspondent,www.theguardian.com,https://www.theguardian.com/world/2025/nov/10/...
4,The Guardian,Terrorist turf war battle in north-eastern Nig...,"Mon, 10 Nov 2025 15:05:02 GMT",Eromo Egbejule West Africa correspondent,www.theguardian.com,https://www.theguardian.com/world/2025/nov/10/...
5,The Guardian,"World must ‘honour 1.5C’, small island states ...","Tue, 11 Nov 2025 21:01:36 GMT","Damien Gayle (now), and Ajit Niranjan (earlier)",www.theguardian.com,https://www.theguardian.com/environment/live/2...
6,The Guardian,Pentagon’s largest warship enters Latin Americ...,"Tue, 11 Nov 2025 17:47:36 GMT","Tiago Rogero, South America correspondent",www.theguardian.com,https://www.theguardian.com/us-news/2025/nov/1...
7,The Guardian,Man and daughter flying hurricane relief suppl...,"Tue, 11 Nov 2025 16:06:40 GMT",Richard Luscombe in Coral Springs,www.theguardian.com,https://www.theguardian.com/us-news/2025/nov/1...
8,The Guardian,Canada no longer measles-free as outbreaks spread,"Tue, 11 Nov 2025 02:23:30 GMT",Associated Press,www.theguardian.com,https://www.theguardian.com/world/2025/nov/11/...
9,The Guardian,Two dead after small plane on hurricane relief...,"Mon, 10 Nov 2025 22:51:52 GMT",Associated Press,www.theguardian.com,https://www.theguardian.com/us-news/2025/nov/1...



✅ Saved combined CSV to: /Users/zzz/Documents/MIE1624/Group_Project/data/vectorstore_news_ai/news_articles_combined_20251111_171931.csv
